# Padrão de Injeção de Bearer Token para Targets do AWS Bedrock AgentCore Gateway

## Visão Geral

Os clientes desejam permitir que um MCP Client (Cyphr, Amazon Q) invoque ferramentas (ex.: funções do Asana) expostas por targets de backend (funções Lambda ou serviços RESTful) através do AgentCore Gateway. Um requisito crítico é injetar um Bearer Token dinâmico do MCP client no target para autenticação com o serviço downstream. O principal desafio é o requisito obrigatório de autenticação de saída no AgentCore Gateway, que pode conflitar com a injeção dinâmica de headers.

O diagrama de arquitetura a seguir mostra a solução que estamos propondo para injetar o token de autenticação (ex.: "Bearer actual-auth-token") sem conflitar com o mecanismo de autenticação de saída do AgentCore.

<img src="./images/token-injection-architecture.png" alt="Diagrama de arquitetura mostrando o padrão de injeção de token" title="Diagrama de arquitetura mostrando o padrão de injeção de token" />

Na solução, o MCP client obtém o token de autenticação a ser utilizado pela ferramenta e o repassa ao AgentCore Gateway como um parâmetro no payload, que então é encaminhado à ferramenta como um header personalizado.

### Implantando os pré-requisitos

Esta solução implanta os seguintes componentes: 
- User pool do Cognito para autenticação de entrada 
- API Gateway com integração Lambda
- API Key para autenticação de saída do AgentCore


In [ ]:
cd prerequisites/agentcore-components

In [ ]:
!bash prereq.sh

## Criando o AgentCore Gateway com API Gateway como target




### Passo 1: Instalar Dependências e Importar 

Antes de começar, vamos instalar os pré-requisitos para este laboratório

In [ ]:
import os
import sys

# Get the directory of the current script
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

print(f"current_dir set to {current_dir}")

# Navigate to the directory containing utils.py (two level up to "04-bearer-token-injection")
utils_dir = os.path.abspath(os.path.join(current_dir, "../.."))

# Add to sys.path
sys.path.insert(0, utils_dir)

import utils

In [ ]:
# Install required packages
%pip install -U -r ../../requirements.txt -q

### Passo 2: Criar o AgentCore Gateway

O código Python a seguir:
- cria o AgentCore Gateway
- adiciona o APIGateway criado anteriormente com API_KEY como Target.

In [ ]:
# Import the functions from the agentcore_gateway_creation module
import sys
import os

# Add the current directory to Python path to import the module
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from agentcore_gateway_creation import create_agentcore_gateway

# Step 1: Create the AgentCore Gateway
print("🚀 Creating AgentCore Gateway...")
gateway = create_agentcore_gateway()
print(f"✅ Gateway created with ID: {gateway['id']}")
print(f"Gateway URL: {gateway['gateway_url']}")

### Passo 3: Adicionar o APIGateway como target no AgentCore Gateway

Primeiro, vamos garantir que o AgentCore Gateway esteja "READY".

In [ ]:
import time
import boto3

# Wait for gateway to be in ACTIVE state
gateway_client = boto3.client("bedrock-agentcore-control")
while True:
    response = gateway_client.get_gateway(gatewayIdentifier=gateway["id"])
    status = response["status"]
    print(f"Gateway status: {status}")
    if status == "READY":
        break
    elif status == "FAILED":
        raise Exception("Gateway creation failed")
    time.sleep(10)  # Wait 10 seconds before checking again

Agora vamos adicionar o gateway target para concluir a configuração:

In [ ]:
from agentcore_gateway_creation import add_gateway_target

# Step 2: Add Gateway Target
print("🎯 Adding Gateway Target...")
add_gateway_target(gateway["id"])
print("✅ Gateway target configuration completed!")

Vamos ver a OpenAPI Spec que é usada para definir este target:

<img src="./images/bearer-token-in-api-spec.png" alt="bearer token na api spec como um header" title="bearer token na api spec como um header" />

O token é definido como um parâmetro de header no schema e encaminhado em um header HTTP: ```X-Asana-Token```. Este é apenas um exemplo, você pode definir um header como este para a sua integração.

O header HTTP é o padrão mais convencional para tokens de autenticação.

### Passo 4: Testar o fluxo de ponta a ponta

Agora vamos testar o fluxo de ponta a ponta onde invocaremos ``` tools/list ``` a partir de um MCP client.

In [ ]:
import os
import boto3
import time
from botocore.exceptions import ClientError
import utils
import uuid
import json
import requests

# Step 1: Get inbound Auth token for MCP client to authenticate with AgentCore
token_url = utils.get_ssm_parameter("/app/asana/demo/agentcoregwy/cognito_token_url")
client_id = utils.get_ssm_parameter("/app/asana/demo/agentcoregwy/machine_client_id")
client_secret = utils.get_cognito_client_secret()
inbound_auth_token = utils.fetch_access_token(client_id, client_secret, token_url)
print("access token received")

# Step 2: Get Gateway config and prepare payload
GATEWAY_MCP_URL = gateway_url = (
    f"https://{utils.get_ssm_parameter('/app/asana/demo/agentcoregwy/gateway_id')}.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"
)

SESSION_ID = str(uuid.uuid4())
headers = {
    "Authorization": f"Bearer {inbound_auth_token}",  # For Inbound Auth
    "Content-Type": "application/json",
    "Mcp-Session-Id": SESSION_ID,
}

# Step 3: Call tools/list to get available tools
list_body = {"jsonrpc": "2.0", "id": "list-1", "method": "tools/list"}

list_response = requests.post(GATEWAY_MCP_URL, headers=headers, json=list_body)
print(f"tools/list Status: {list_response.status_code}")
print("Available Tools:")
print(json.dumps(list_response.json(), indent=2))

### Passo 5: Injeção de Bearer Token do MCP client para o target do AgentCore Gateway

Agora vamos passar o Bearer Token ``` Bearer ASANA-TOKEN ``` nos parâmetros (você substituiria ``` ASANA-TOKEN ``` pelo token real que seu MCP client teria obtido) e invocar a ferramenta ```AgentCoreGwyAPIGatewayTarget___asanaInvoke``` da resposta de ```tools/list``` acima. 

In [ ]:
# Step 1: Get inbound Auth token for MCP client to authenticate with AgentCore
token_url = utils.get_ssm_parameter("/app/asana/demo/agentcoregwy/cognito_token_url")
client_id = utils.get_ssm_parameter("/app/asana/demo/agentcoregwy/machine_client_id")
client_secret = utils.get_cognito_client_secret()
inbound_auth_token = utils.fetch_access_token(client_id, client_secret, token_url)
print("access token received")

# Step 2: Get Gateway config and prepare payload
GATEWAY_MCP_URL = gateway_url = (
    f"https://{utils.get_ssm_parameter('/app/asana/demo/agentcoregwy/gateway_id')}.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"
)

SESSION_ID = str(uuid.uuid4())
headers = {
    "Authorization": f"Bearer {inbound_auth_token}",  # For Inbound Auth
    "Content-Type": "application/json",
    "Mcp-Session-Id": SESSION_ID,
}

# Step 3: Call tools/call with Asana token in arguments (will forward as header per updated schema)
call_body = {
    "jsonrpc": "2.0",
    "id": "call-1",
    "method": "tools/call",
    "params": {
        "name": "AgentCoreGwyAPIGatewayTarget___asanaInvoke",  # Exact name from tools/list
        "arguments": {
            "tool_name": "createTask",
            "X-Asana-Token": "Bearer ASANA-TOKEN",  # Actual Asana bearer token here (forwards to header)
            "name": "Test Task from MCP",
            "notes": "This is a test description",
            "project": "your-project-gid",  # Replace with actual Project GID
        },
    },
}

call_response = requests.post(GATEWAY_MCP_URL, headers=headers, json=call_body)
print(f"tools/call Status: {call_response.status_code}")
print("Tool Call Response:")
print(json.dumps(call_response.json(), indent=2))

### Passo 6: Verificando a injeção de token

Vamos verificar a injeção de token conferindo os logs no CloudWatch. Navegue até a stack do Cloudformation ```AsanaIntegrationStackInfra``` criada nos pré-requisitos: 

```CloudFormation -> Stacks -> AsanaIntegrationStackInfra``` e procure pela função Lambda conforme mostrado abaixo:
<img src="./images/AgentCoreGwyAsanaIntegrationDemo_function_resource.png" alt="Recurso da função AgentCoreGwyAsanaIntegrationDemo" title="Recurso da função AgentCoreGwyAsanaIntegrationDemo"/> 


Abra a função Lambda em uma janela separada e clique em ```View CloudWatch Logs``` conforme mostrado na captura de tela abaixo: <img src="./images/AgentCoreGwyAsanaIntegrationDemo_function_log_group.png" alt="Grupo de logs da função AgentCoreGwyAsanaIntegrationDemo" title="Grupo de logs da função AgentCoreGwyAsanaIntegrationDemo"/>  


A captura de tela a seguir mostra os logs da invocação da ferramenta ```AgentCoreGwyAPIGatewayTarget___asanaInvoke``` via AgentCore Gateway. Ela mostra que o ``` Bearer token ``` foi passado do MCP client para o AgentCore Gateway até chegar à função Lambda no header ``` X-Asana-Token ``` conforme definido na OpenAPI spec. 

<img src="./images/Lambda-log-received-headers.png" alt="Bearer token injetado como header" title="Bearer token injetado como header" />


Isso demonstra que o token de autenticação obtido pelo MCP client pode ser utilizado pela função Lambda (ou outra ferramenta) para se autenticar com integrações SaaS de terceiros como o Asana sem conflitar com a autenticação de saída do AgentCore.


O restante do payload do MCP client está disponível para a função Lambda no body do evento conforme mostrado abaixo:
<img src="./images/Lambda-log-received-body.png" alt="body do payload" title="body do payload" />

## Limpeza

### Excluir o AgentCore Gateway

In [ ]:
import utils
import boto3

gateway_client = boto3.client("bedrock-agentcore-control")

gateway_id = "agentcore-gw-asana-integration"
utils.delete_gateway(gateway_client, gateway_id)

Deleting all targets for gateway agentcore-gw-asana-integration-gyhxiv6rt5


### Excluir as stacks do Cloudformation

In [ ]:
cd 02-AgentCore-gateway/04-bearer-token-injection

In [ ]:
!bash clean_up.sh